In [1]:
import numpy as np
import pandas as pd

df = pd.read_pickle("models/results.pkl")
TYPES = {"visual", "auditory", "TMS", "TMS_st"}

In [2]:
def mad(x):
    x = np.asarray(x)
    med = np.median(x)
    return np.median(np.abs(x - med))

def rms(x):
    x = np.asarray(x, dtype=float)
    return np.sqrt(np.mean(x**2))

def normalize_by_rms(vec, start):
    vec = np.asarray(vec, dtype=float)
    r = rms(vec[start:])
    return vec / r if r > 0 else vec.copy()

def per_trial_corr(X, template, post_start):
    X = np.asarray(X)[:, post_start:]
    t = np.asarray(template)[post_start:]
    A = np.vstack([X, t])
    C = np.corrcoef(A)
    return C[:-1, -1]

def summarize_corr2(r):
    c2 = np.square(r)
    return np.median(c2), mad(c2)

# Filter to same-type rows
df_same = df[
    (df['stimulation_type_test'] == df['stimulation_type_train']) &
    (df['stimulation_type_test'].isin(TYPES))
].copy()
if df_same.empty:
    raise ValueError("No rows with identical test/train and desired types.")

# Per-row aggregates
df_same['trial_sums']   = df_same['X'].apply(lambda X: np.sum(X, axis=0))
df_same['trial_counts'] = df_same['X'].apply(len)

# Compute within-subject template and its RMS-normalized version (over post)
df_same['mu_within'] = df_same.apply(lambda r: r['trial_sums'] / r['trial_counts'], axis=1)
df_same['mu_within_norm'] = df_same.apply(lambda r: normalize_by_rms(r['mu_within'], start=int(r['pre_end'])), axis=1)

# Option: choose weights for across-subject averaging
# weights = 1.0 -> equal subject weight; or use trial_counts for trial-weighted LOSO
df_same['loso_weight'] = 1.0  # or df_same['trial_counts']

# Pre-index by type for fast LOSO construction
type_groups = {typ: idxs.index.to_list()
               for typ, idxs in df_same.groupby('stimulation_type_test')}

def compute_row(row):
    idx = row.name
    typ = row['stimulation_type_test']
    pre_end = int(row['pre_end'])
    X = np.asarray(row['X'])

    # Within-subject correlations (template already normalized for reporting)
    mu_within_norm = row['mu_within_norm']
    mu_within = row['mu_within']
    r_within = per_trial_corr(X, mu_within, pre_end)
    w_med, w_mad = summarize_corr2(r_within)

    # LOSO: average RMS-normalized templates from other subjects of same type
    indices = type_groups[typ]
    others = [j for j in indices if j != idx]
    if len(others) == 0:
        l_med = l_mad = np.nan
    else:
        num = 0.0
        den = 0.0
        for j in others:
            wj = df_same.at[j, 'loso_weight']
            mu_j = df_same.at[j, 'mu_within_norm']  # already RMS-normalized over post
            num = num + wj * mu_j
            den = den + wj
        mu_loso = num / den
        # Optional: normalize mu_loso again (not necessary for r, but keeps unit RMS)
        # mu_loso = normalize_by_rms(mu_loso, start=pre_end)

        r_loso = per_trial_corr(X, mu_loso, pre_end)
        l_med, l_mad = summarize_corr2(r_loso)

    return pd.Series(dict(
        corr2_within_median=w_med,
        corr2_within_mad=w_mad,
        corr2_loso_median=l_med,
        corr2_loso_mad=l_mad,
    ))

df_corr = df_same.join(df_same.apply(compute_row, axis=1))

# Aggregate by type
def agg_mad(s): return mad(s)

agg = (
    df_corr
    .groupby('stimulation_type_test', as_index=False)
    .agg(
        within_count=('corr2_within_median', 'count'),
        within_median=('corr2_within_median', 'median'),
        within_mad=('corr2_within_median', agg_mad),
        loso_count=('corr2_loso_median', 'count'),
        loso_median=('corr2_loso_median', 'median'),
        loso_mad=('corr2_loso_median', agg_mad),
    )
)

print("\n=== Template–trial correlations squared (median ± MAD); subject templates RMS-normalized before LOSO averaging ===")
print(agg.to_string(index=False))

for typ, grp in df_corr.groupby('stimulation_type_test'):
    w_m = np.median(grp['corr2_within_median']); w_s = mad(grp['corr2_within_median'])
    l_m = np.median(grp['corr2_loso_median']);   l_s = mad(grp['corr2_loso_median'])
    print(f"\nType: {typ}")
    print(f"  Within-subject r^2: {w_m:.2f} ± {w_s:.2f}")
    print(f"  LOSO template r^2:  {l_m:.2f} ± {l_s:.2f}")



=== Template–trial correlations squared (median ± MAD); subject templates RMS-normalized before LOSO averaging ===
stimulation_type_test  within_count  within_median  within_mad  loso_count  loso_median  loso_mad
                  TMS            52       0.368145    0.211001          52     0.171787  0.116711
               TMS_st            52       0.467130    0.207067          52     0.251177  0.162013
             auditory            92       0.438418    0.080736          92     0.283232  0.111341
               visual           219       0.496092    0.111458         219     0.349225  0.148271

Type: TMS
  Within-subject r^2: 0.37 ± 0.21
  LOSO template r^2:  0.17 ± 0.12

Type: TMS_st
  Within-subject r^2: 0.47 ± 0.21
  LOSO template r^2:  0.25 ± 0.16

Type: auditory
  Within-subject r^2: 0.44 ± 0.08
  LOSO template r^2:  0.28 ± 0.11

Type: visual
  Within-subject r^2: 0.50 ± 0.11
  LOSO template r^2:  0.35 ± 0.15
